# Population and Dwelling Counts Exploration

**Objective:** Explore population and dwelling counts for Canada, the provinces, and the territories using a beginner-friendly workflow.

**Dataset description:** This dataset comes from Statistics Canada's 2021 Census table 98-10-0001-01. Each row is one geography: Canada, a province, or a territory. The columns include population, dwelling counts, land area, and population density.

**What does each row represent?**
Each row represents one geography in Canada for the 2021 Census.

**Data source(s):**
- Table page: https://www150.statcan.gc.ca/t1/tbl1/en/tv.action?pid=9810000101
- ZIP download: https://www150.statcan.gc.ca/n1/tbl/csv/98100001-eng.zip

**Date / Author:** 2026-03-15 / Codex


## Step 1: Import our libraries

This cell imports the tools we need for the rest of the notebook.

- `pandas` (`pd`) helps us work with tables of data.
- A **DataFrame** is a table object in pandas.
- We often call the table `df`, which is short for "data frame."
- `plotly.express` (`px`) helps us make charts.
- `Path`, `urlretrieve`, and `ZipFile` help us save and extract the dataset files.


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile

import pandas as pd
import plotly.express as px


## Step 2: Set the download link and file paths

This cell saves the web address for the dataset and chooses where the ZIP file and CSV file will go in our project folder.

We use `Path(...)` so the file paths are clear and easy to reuse.


In [ ]:
download_url = "https://www150.statcan.gc.ca/n1/tbl/csv/98100001-eng.zip"

dataset_dir = Path("datasets/population-dwelling-counts-2021")
zip_path = dataset_dir / "98100001-eng.zip"
csv_path = dataset_dir / "98100001.csv"
metadata_path = dataset_dir / "98100001_MetaData.csv"

dataset_dir.mkdir(parents=True, exist_ok=True)

zip_path, csv_path


## Step 3: Download the ZIP file

This cell downloads the StatCan ZIP file into the local `datasets/` folder.

The `urlretrieve(...)` function saves the file directly to the location we choose.


In [ ]:
urlretrieve(download_url, zip_path)
zip_path.exists(), zip_path.stat().st_size


## Step 4: Look inside the ZIP archive

This cell lists the files inside the ZIP so we can confirm the CSV file name before extracting it.


In [ ]:
with ZipFile(zip_path) as zipped_file:
    zipped_file.namelist()


## Step 5: Extract the ZIP file

This cell unpacks the files so the CSV and metadata become regular files we can open.


In [ ]:
with ZipFile(zip_path) as zipped_file:
    zipped_file.extractall(dataset_dir)

sorted(path.name for path in dataset_dir.iterdir())


## Step 6: Load the CSV and preview the first rows

This cell reads the CSV into a DataFrame named `df`.

In `df.head()`, `head()` is a **method**, which means it performs an action.


In [ ]:
df = pd.read_csv(csv_path)
df.head()


## Step 7: Display the column names

This cell shows all the column names so we can see what information is included.

In `df.columns`, `columns` is a **property**, which means it gives information about the DataFrame.


In [ ]:
display(df.columns)


## Step 8: Rename the columns to shorter names

This cell creates a cleaned copy of the table with shorter column names.

- `copy()` makes a separate DataFrame so we do not accidentally change the original.
- Shorter names make the next steps easier to read.


In [ ]:
cleaned_df = df.copy()

cleaned_df.columns = [
    "ref_date",
    "geo",
    "dguid",
    "coordinate",
    "population_2021",
    "population_2021_symbol",
    "population_2016",
    "population_2016_symbol",
    "population_pct_change",
    "population_pct_change_symbol",
    "total_private_dwellings_2021",
    "total_private_dwellings_2021_symbol",
    "total_private_dwellings_2016",
    "total_private_dwellings_2016_symbol",
    "total_private_dwellings_pct_change",
    "total_private_dwellings_pct_change_symbol",
    "occupied_private_dwellings_2021",
    "occupied_private_dwellings_2021_symbol",
    "occupied_private_dwellings_2016",
    "occupied_private_dwellings_2016_symbol",
    "occupied_private_dwellings_pct_change",
    "occupied_private_dwellings_pct_change_symbol",
    "land_area_sq_km_2021",
    "land_area_sq_km_2021_symbol",
    "population_density_2021",
    "population_density_2021_symbol",
]

cleaned_df.head()


## Step 9: Convert numeric columns to numbers

This cell changes number columns from text into numeric values so we can sort and graph them.

If a value cannot be converted, `errors="coerce"` will turn it into `NaN` instead of causing an error.
A `NaN` means a value is missing.


In [ ]:
numeric_columns = [
    "coordinate",
    "population_2021",
    "population_2016",
    "population_pct_change",
    "total_private_dwellings_2021",
    "total_private_dwellings_2016",
    "total_private_dwellings_pct_change",
    "occupied_private_dwellings_2021",
    "occupied_private_dwellings_2016",
    "occupied_private_dwellings_pct_change",
    "land_area_sq_km_2021",
    "population_density_2021",
]

for column in numeric_columns:
    cleaned_df[column] = pd.to_numeric(cleaned_df[column], errors="coerce")

cleaned_df.dtypes


## Step 10: Show the main comparison columns

This cell keeps a smaller set of columns that are easy to compare in class.


In [ ]:
summary_df = cleaned_df[[
    "geo",
    "population_2021",
    "population_2016",
    "population_pct_change",
    "total_private_dwellings_2021",
    "land_area_sq_km_2021",
    "population_density_2021",
]]

summary_df


## Step 11: Make a bar chart of 2021 population

This cell builds a horizontal bar chart so we can compare population sizes.

We remove the Canada total first so the province and territory values are easier to compare.


In [ ]:
province_territory_df = summary_df[summary_df["geo"] != "Canada"].copy()
chart_df = province_territory_df.sort_values("population_2021")

fig = px.bar(
    chart_df,
    x="population_2021",
    y="geo",
    orientation="h",
    title="Population in 2021 by province and territory",
    labels={"population_2021": "Population", "geo": "Geography"},
)
fig.show()


## Step 12: Find the fastest population growth

This cell sorts by percentage change from 2016 to 2021.

Sorting helps us identify which province or territory grew the fastest.


In [ ]:
province_territory_df.sort_values("population_pct_change", ascending=False)[[
    "geo",
    "population_pct_change",
    "population_2021",
]]


## Step 13: Make a population density chart

This cell compares how crowded each province or territory is using population density.

Population density means the number of people per square kilometre.


In [ ]:
density_chart_df = province_territory_df.sort_values("population_density_2021")

fig = px.bar(
    density_chart_df,
    x="population_density_2021",
    y="geo",
    orientation="h",
    title="Population density in 2021 by province and territory",
    labels={"population_density_2021": "People per square kilometre", "geo": "Geography"},
)
fig.show()


## Step 14: Read the metadata file

This cell loads the metadata table, which explains the cube, the variables, and the source.


In [ ]:
metadata_df = pd.read_csv(metadata_path)
metadata_df.head()


## Step 15: Reflection

This dataset is useful for questions such as:
- Which province has the largest population?
- Which province or territory had the fastest growth from 2016 to 2021?
- How does land area compare with population density?

A limitation is that this table only covers Canada, provinces, and territories. It does not include smaller regions such as cities or census divisions.


---
### Open In Colab
[Open this notebook in Google Colab](https://colab.research.google.com/github/pbeens/ICD2O-OSC-2026-Project-Plan-Team-2/blob/main/notebooks/zip/population-dwelling-counts-canada-provinces-territories-exploration.ipynb)
